# ServerWatch demo commands

Copy each cell into the terminal; replace `ELASTIC_IP` with the instance address.
The talking points are in the [presentation guide](presentation-guide.md).

## 0. Create the SSH keys (G2)
Creates two key pairs on your computer: one for the `ubuntu` admin login and one for the `junior` account. The private keys stay on your computer; only the `.pub` files are shared.

In [ ]:
ssh-keygen -t ed25519 -C serverwatch-ubuntu -f ~/.ssh/serverwatch_ec2
ssh-keygen -t ed25519 -C serverwatch-junior -f ~/.ssh/junior_key
ls -l ~/.ssh/serverwatch_ec2* ~/.ssh/junior_key*

Prints the admin public key, to paste into AWS under **EC2 → Key pairs → Import key pair**.

In [ ]:
cat ~/.ssh/serverwatch_ec2.pub

## 1. Launch the instance (G1)
In the AWS console: Ubuntu 24.04 LTS, a free-tier type, the imported key pair, and a security group with **SSH 22 from My IP** only. Then allocate an **Elastic IP** and associate it with the instance.

## 2. Connect with SSH (G1, G2)
Logs in as `ubuntu` with the imported key. Answer `yes` to the fingerprint question; SSH then remembers this server.

In [ ]:
ssh -i ~/.ssh/serverwatch_ec2 ubuntu@ELASTIC_IP

## 3. Clone the repository (G4)
Installs Git and clones the project into the `ubuntu` user's home folder, so that user owns the code.

In [ ]:
sudo apt-get update && sudo apt-get install -y git
git clone https://github.com/EmirU116/server-api.git ~/serverwatch
cd ~/serverwatch && git log -1 --oneline

## 4. Copy the setup script with SCP (G3)
Sends `setup.sh` to the server over SSH. Run on your computer, from the project folder.

In [ ]:
scp -i ~/.ssh/serverwatch_ec2 setup.sh ubuntu@ELASTIC_IP:~/setup.sh

## 5. Save the Discord webhook (G6)
Stores the webhook in a file only you can read; the prompt hides what you paste.

In [ ]:
umask 077
install -d -m 700 "$HOME/.config/serverwatch-alerts"
if IFS= read -rsp 'Discord webhook URL: ' webhook; then
    printf '%s\n' "$webhook" > "$HOME/.config/serverwatch-alerts/discord-webhook"
fi
unset webhook
printf '\n'

## 6. Run setup (G3)
Installs everything: PostgreSQL, the API and collector services, Nginx, Cron backups, Docker, the two accounts, the Discord timer and the `sw-*` aliases.

In [ ]:
export SERVERWATCH_SENIOR_USER=senior
export SERVERWATCH_JUNIOR_USER=junior
export SERVERWATCH_DISCORD_WEBHOOK_FILE="$HOME/.config/serverwatch-alerts/discord-webhook"
SERVERWATCH_PROJECT_DIR="$HOME/serverwatch" bash ~/setup.sh ELASTIC_IP

## 7. Second account with its own key (G2)
Gives the `junior` account its own key. First print the public key on your computer:

In [ ]:
cat ~/.ssh/junior_key.pub

Then on EC2, paste it into the account's `authorized_keys` in nano.

In [ ]:
sudo install -d -o junior -g junior -m 700 /home/junior/.ssh
sudo -u junior nano /home/junior/.ssh/authorized_keys
sudo chmod 600 /home/junior/.ssh/authorized_keys

Then from a second terminal or computer, log in as `junior` and show its groups.

In [ ]:
ssh -i ~/.ssh/junior_key junior@ELASTIC_IP
id

## 8. Verify the deployment (G7)
Read-only checks of the services, database and Nginx; expect 8 passed, 0 failed.

In [ ]:
cd ~/serverwatch
bash verify.sh ELASTIC_IP

## 9. Nginx by hand (G8)
Opens the reverse-proxy config in nano, checks the syntax, and reloads Nginx. Then add HTTP 80 in the security group and open `http://ELASTIC_IP/docs`.

In [ ]:
cd /etc/nginx/sites-enabled
ls -l
sudo nano serverwatch.conf
sudo nginx -t
sudo systemctl reload nginx

## 10. systemd and the database (G7, G9)
Shows the API and collector running under systemd, and that the API and PostgreSQL only listen on 127.0.0.1.

In [ ]:
sw-status
systemctl cat serverwatch-api
sudo ss -ltnp | grep -E ':5432|:8000'

## 11. Cron backup (G5)
Shows the daily backup schedule; edit it to the next minute, then check the log and the backup files. Put `15 3` back afterwards.

In [ ]:
sudo cat /etc/cron.d/serverwatch-backup
sudoedit /etc/cron.d/serverwatch-backup

In [ ]:
sudo journalctl -t serverwatch-backup -n 10 --no-pager
sudo ls -lt /var/backups/serverwatch

Restore into a separate database with the block in the [EC2 guide](ec2-manual-setup.md#verify-cron-backups-and-restore-g5) and compare row counts.

## 12. Discord alert (G6)
Lowers the disk threshold below current usage so the timer posts to Discord within a minute.

In [ ]:
cd ~/serverwatch
sw-timers
df -h /
bash infrastructure/bootstrap/setup_discord_alerts.sh "$HOME/.config/serverwatch-alerts/discord-webhook" 25

After the message arrives, put the normal 90% threshold back.

In [ ]:
bash infrastructure/bootstrap/setup_discord_alerts.sh "$HOME/.config/serverwatch-alerts/discord-webhook" 90

## 13. Docker Compose (VG1)
Builds the images and starts db → migrate → api → web; `migrate` showing `Exited (0)` is success.

In [ ]:
cd ~/serverwatch
bash scripts/deployment/compose_host.sh start
bash scripts/deployment/compose_host.sh ps -a

Sends this host's metrics to the Compose API; stop with Ctrl+C after about 20 seconds.

In [ ]:
bash scripts/deployment/compose_host.sh collect

## 14. Data survives container recreation (VG1)
Counts the rows, removes the containers (the volume stays), starts them again, and counts again; both counts must match.

In [ ]:
bash scripts/deployment/compose_host.sh exec db psql -U postgres -d serverwatch -c "SELECT count(*) FROM metrics;"
bash scripts/deployment/compose_host.sh down
bash scripts/deployment/compose_host.sh up -d --wait
bash scripts/deployment/compose_host.sh exec db psql -U postgres -d serverwatch -c "SELECT count(*) FROM metrics;"

## 15. Domain and HTTPS (VG2)
First point both Namecheap A records at the Elastic IP and add HTTPS 443 in the security group. This checks that DNS points here.

In [ ]:
getent hosts serverwatch.dev www.serverwatch.dev

Gets a Let's Encrypt certificate with Certbot, turns on HTTPS and checks renewal. For rehearsals put `SERVERWATCH_CERTBOT_STAGING=1` in front to save real certificates.

In [ ]:
cd ~/serverwatch
bash infrastructure/bootstrap/setup_https.sh serverwatch.dev ELASTIC_IP

Shows the certificate, the renewal timer and the HTTPS Nginx site. Then open `https://serverwatch.dev`.

In [ ]:
sudo certbot certificates
systemctl list-timers certbot.timer
sudo nano /etc/nginx/sites-enabled/serverwatch-domain.conf

## 16. Optional: deploy after CI (VG4 bonus)
After pushing a change and watching *ServerWatch deploy* in GitHub Actions, shows which commit is running in Compose. Needs the setup in [continuous deployment](continuous-deployment.md).

In [ ]:
cat ~/.config/serverwatch-delivery/deployed-commit
bash scripts/deployment/compose_host.sh ps

## Afterwards
Watch the recording, check the link, then stop the instance and release the Elastic IP in the AWS console.